# 02 · LoRA From Scratch

The module, built in plain PyTorch: a **frozen** weight $W$ plus a down-projection
$A$ (to rank $r$) and an up-projection $B$ (back), scaled by $\alpha/r$, with $B$
**zero-initialized** so the adapter starts as a no-op. We verify in code that at
step 0 the LoRA layer equals the base, that gradients flow **only** to $A$ and $B$,
and that the trainable count is exactly $2 d r$.

In [1]:
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))
print('repo root:', _p)

repo root: /workspaces/lora_101


In [2]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from src.config import get_settings
from src.lora.layer import LoRALinear
from src.lora.params import lora_parameter_count, full_parameter_count, count_parameters
from src.lora.inject import inject_lora
from src.tasks.toy import LinearModel
from src.visualisation import save_figure
from src.results import save_json

s = get_settings()
torch.manual_seed(s.seed)
d, r, alpha = 64, s.lora.r, s.lora.alpha
base = nn.Linear(d, d)
lora = LoRALinear.from_linear(base, r=r, alpha=alpha)
x = torch.randn(8, d)
max_diff = (lora(x) - base(x)).abs().max().item()
print(f'at init, max|lora(x) - base(x)| = {max_diff:.2e}  (B is zero-init -> no-op)')
print('nonzeros in B:', int(torch.count_nonzero(lora.lora_B)))

at init, max|lora(x) - base(x)| = 0.00e+00  (B is zero-init -> no-op)
nonzeros in B: 0


## Gradients flow only to the adapter

In [3]:
probe = LoRALinear(d, d, r=r, alpha=alpha)
with torch.no_grad():
    probe.lora_B.normal_()  # nonzero B so A also receives gradient
probe(x).pow(2).sum().backward()
grad_flags = {
    'weight_has_grad': probe.weight.grad is not None,
    'bias_has_grad': probe.bias.grad is not None,
    'lora_A_has_grad': probe.lora_A.grad is not None,
    'lora_B_has_grad': probe.lora_B.grad is not None,
}
print(grad_flags)

{'weight_has_grad': False, 'bias_has_grad': False, 'lora_A_has_grad': True, 'lora_B_has_grad': True}


## The trainable count is exactly $2dr$

In [4]:
model = LinearModel(d, d, bias=False)
inject_lora(model, target_modules=['linear'], r=r, alpha=alpha)
trainable, total = count_parameters(model)
formula = lora_parameter_count(d, d, r)
print(f'trainable={trainable}  |  2*d*r={2 * d * r}  |  formula={formula}')
assert trainable == formula == 2 * d * r

trainable=1024  |  2*d*r=1024  |  formula=1024


## The cost gap grows with dimension (PNG + JSON)

In [5]:
ds = [64, 256, 1024, 4096]
full_counts = [full_parameter_count(dd, dd) for dd in ds]
lora_counts = [lora_parameter_count(dd, dd, r) for dd in ds]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ds, full_counts, marker='o', label='full (d*d)')
ax.plot(ds, lora_counts, marker='s', label=f'LoRA (2*d*r, r={r})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('layer dimension d'); ax.set_ylabel('trainable parameters')
ax.set_title('Trainable parameters: full vs LoRA'); ax.legend()
png = save_figure(fig, '02_param_cost')
json_path = save_json({
    'rank': r, 'alpha': alpha, 'dim_demo': d,
    'zero_init_max_diff': max_diff,
    'gradient_flow': grad_flags,
    'trainable_params': trainable, 'two_d_r': 2 * d * r,
    'cost_curve': {'d': ds, 'full': full_counts, 'lora': lora_counts,
                   'lora_fraction': [lc / fc for lc, fc in zip(lora_counts, full_counts)]},
}, '02_lora_from_scratch')
print('saved:', png.name, 'and', json_path.name)

saved: 02_param_cost.png and 02_lora_from_scratch.json


## Takeaway
Nothing of the base moves. Only $A$ and $B$ train, their count is exactly $2dr$,
and the fraction of the full matrix shrinks as the layer grows — the arithmetic
behind 'under one percent'.